# Lab: Building an MCP Server from Scratch

## What is MCP?

**MCP** stands for **Model Context Protocol**.

It is a standard way for AI applications to connect to external tools, data, and workflows.

You can think of MCP as a **common interface** between:

- an **AI client** (such as a chatbot or LLM application)
- and an **external capability provider** (an MCP server)

Instead of hardcoding every tool directly inside the chatbot, MCP allows tools to be exposed by a separate server in a standardized way.

---

## MCP Server vs MCP Client

### MCP Server
An MCP server is a program that exposes capabilities such as:

- tools
- resources
- workflows

In this lab, our MCP server will expose tools such as:

- `get_course_info(course_name)`
- `calculate_final_grade(assignments, midterm, final_exam)`
- `days_until_deadline(deadline_date)`

### MCP Client
The MCP client is the application that connects to the MCP server and uses its tools.

In this lab, the **LLM application** will act as the client.

---

## Why Use MCP?

Without MCP:

- tools are tightly coupled to one application
- integration is harder to reuse
- every AI app may need its own custom connection logic

With MCP:

- tools can be reused across different AI clients
- the interface is standardized
- the LLM can discover and call tools more cleanly

---

## Lab Scenario

You will build a small **Course Assistant MCP Server**.

This server will provide information related to courses and simple academic utilities.

Then, you will connect the server to an LLM and ask questions such as:

- *What is the course info for AI?*
- *If I scored 18 in assignments, 25 in midterm, and 40 in final, what is my total?*
- *How many days are left until 2026-05-20?*

The LLM should decide to call the right MCP tool.


## Architecture of This Lab

The flow looks like this:

```text
User Prompt -> LLM Client -> MCP Server -> Tool Execution -> Result -> LLM Response
```

More specifically:

1. The user asks the LLM a question.
2. The LLM sees that an MCP server is available.
3. The LLM discovers the tools exposed by the server.
4. The LLM selects the correct tool.
5. The server executes the tool and returns the result.
6. The LLM uses that result to answer the user.


## Step 1 — Install the Required Packages


In [ ]:
!pip install fastmcp openai


## Step 2 — Build the MCP Server from Scratch

We will now write a Python MCP server.

This server will:

- create a FastMCP server instance
- define a small in-memory course dataset
- expose three tools
- run using SSE transport

In a real deployment, the server would be hosted somewhere reachable by the LLM client.


In [ ]:
server_code = r'''
from fastmcp import FastMCP
from datetime import datetime

mcp = FastMCP(name="Course Assistant MCP Server")

COURSES = {
    "ai": {
        "title": "Introduction to Artificial Intelligence",
        "instructor": "Dr. Sara",
        "credit_hours": 3
    },
    "ml": {
        "title": "Machine Learning",
        "instructor": "Dr. Omar",
        "credit_hours": 3
    },
    "db": {
        "title": "Database Systems",
        "instructor": "Dr. Mona",
        "credit_hours": 4
    }
}

@mcp.tool()
def get_course_info(course_name: str) -> dict:
    """Return information about a course by its short name."""
    course_name = course_name.strip().lower()
    if course_name in COURSES:
        return COURSES[course_name]
    return {"error": "Course not found"}

@mcp.tool()
def calculate_final_grade(assignments: float, midterm: float, final_exam: float) -> dict:
    """Calculate total grade from three assessment components."""
    total = assignments + midterm + final_exam
    return {
        "assignments": assignments,
        "midterm": midterm,
        "final_exam": final_exam,
        "total": total
    }

@mcp.tool()
def days_until_deadline(deadline_date: str) -> dict:
    """Return number of days between today and a deadline. Format: YYYY-MM-DD"""
    today = datetime.today().date()
    deadline = datetime.strptime(deadline_date, "%Y-%m-%d").date()
    diff = (deadline - today).days
    return {
        "today": str(today),
        "deadline": str(deadline),
        "days_remaining": diff
    }

if __name__ == "__main__":
    mcp.run(transport="sse", host="0.0.0.0", port=8000)
'''

with open("server.py", "w", encoding="utf-8") as f:
    f.write(server_code)

print("server.py has been created successfully.")


### Explanation of the Server Code

Let us break down what we just built:

- `FastMCP(...)` creates the MCP server.
- `@mcp.tool()` marks a Python function as a tool that can be exposed to clients.
- `COURSES` is a simple in-memory dataset.
- `mcp.run(...)` starts the server.

This means that any MCP-compatible client can connect to the server, inspect the available tools, and call them.


## Step 3 — View the Generated Server File

Use the next cell to display the contents of `server.py`.


In [ ]:
with open("server.py", "r", encoding="utf-8") as f:
    print(f.read())


## Step 4 — Run the MCP Server

Run the following command in your terminal:

```bash
python server.py
```

When the server starts successfully, it should listen on port `8000` and expose its SSE endpoint.

> Important: For an external LLM service to call your MCP server, the server must be reachable through a public URL, not only `localhost`.


## Step 5 — Expose the Server Publicly

If your server runs only on your own machine, an external API cannot reach it directly.

So for the LLM connection step, you need a **public URL**.

Common options:

- Replit
- Railway
- Render
- any host that gives you a public URL

After deployment, your server URL may look like this:

```text
https://your-server-domain.example/sse/
```

You will place that public URL into the client code in the next section.


## Step 6 — Call the MCP Server Using an LLM

Now we create a client example that uses the **OpenAI Responses API**.

The LLM will:

- connect to the remote MCP server
- list its available tools
- choose the correct tool when needed
- use the tool result in its final response


In [ ]:
client_code = r'''
from openai import OpenAI

client = OpenAI(api_key="YOUR_OPENAI_API_KEY")

response = client.responses.create(
    model="gpt-5.4",
    input="What is the course info for AI and how many credit hours is it?",
    tools=[
        {
            "type": "mcp",
            "server_label": "course_server",
            "server_url": "YOUR_PUBLIC_MCP_SERVER_URL",
            "allowed_tools": [
                "get_course_info",
                "calculate_final_grade",
                "days_until_deadline"
            ],
            "require_approval": "never"
        }
    ]
)

print(response.output_text)
'''

with open("client_example.py", "w", encoding="utf-8") as f:
    f.write(client_code)

print("client_example.py has been created successfully.")


### Explanation of the LLM Client Code

- `model="gpt-5.4"` selects the model.
- `input=...` is the user's prompt.
- `tools=[...]` tells the model that a remote MCP server is available.
- `type="mcp"` means the tool source is an MCP server.
- `server_url` points to the public MCP server URL.
- `allowed_tools` limits the model to those tools only.

When the model receives the question, it can inspect the server tools and decide whether to call one of them.


##  Task Description

Modify your MCP server by adding a new tool and demonstrate that the LLM can use it correctly.

### 1 Add a New Tool to Your MCP Server

Implement the following tool:

```python
@mcp.tool()
def gpa_to_letter(gpa: float) -> dict:
    """
    Convert GPA (0–4 scale) to letter grade.
    """
```

#### Expected Behavior:

| GPA Range   | Letter |
|------------|--------|
| 3.7 – 4.0   | A      |
| 3.0 – 3.69  | B      |
| 2.0 – 2.99  | C      |
| 1.0 – 1.99  | D      |
| < 1.0       | F      |



### 2 Update Your LLM Client

Add the new tool to the `allowed_tools` list

### 3 Test with the LLM

Run at least **3 prompts** such as:

- "Convert GPA 3.8 to a letter grade"
- "If my GPA is 2.5, what is my grade?"
- "What letter grade corresponds to GPA 1.2?"